# Generate Figure in style of Figure 2 of Kaiser et al. 2020 ( log(K/Ca) vs. log(Na/Ca) ) but for any element combo

This is the same message at the beginning of all jupyter notebooks in this directory. 

If you don't have the below packages, you obviously need to install them for this to work. If it doesn't work still it's extremely likely you have an outdated version of one of the packages. Alternatively, some of the histogram functions actually rely on not being the most recent version because they changed from "normed" to something else from my recollection. Or perhaps it was the other way. I am aware this was poor decision-making, but it works (if you use the right version). ¯\\_(ツ)_/¯

Also pretty much all of these commands are copied and pasted from another Jupyter notebook I made but contained tons of tries at doing this stuff (and unrelated efforts) so that's why a lot of the variables seem unnecessary to use.

Ok, we basically need to follow the convention of Swan et al. 2019 and create arrows whose length is equal to some multiple of the "e-folding" time. 

## 2021-10-08 The flexibility for modelers, atm_type, and overshoot have been added. Also, the decreasing phase arrow error has been corrected.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1636555364.491528
two_arm_compare_SDSS
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


In [2]:
plt.show()

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/z_plots_for_Hollands/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [5]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
lodders_abund_file='Lodders2009_solarsystem_abundances.csv'
#solar_system_object_file='solar_system_body_abundances.csv'
#solar_system_object_file='solar_system_body_abundances_mgfe_fixed.csv'
#solar_system_object_file='solar_system_body_abundances_sea_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_show_name_added.csv'
solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'

In [6]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit


In [7]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]

In [8]:
color_dict={
    'WDJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5',
    'SDSSJ1636+1619':'pink',
    'WDJ2317+1830':'orange',
    'WDJ1824+1213':'g',
    'LHS2534':'b'
}
step_dict={
    'WDJ1644-0449':5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':5,
    'SDSSJ1636+1619':5,
    'WDJ2317+1830':5,
    'WDJ1824+1213':5,
    'LHS2534':5
}

t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=False

In [9]:
def add_arrow(line, position=None, direction='right', size=20, color=None, slope=None):
    """
    add an arrow to a line.

    line:       Line2D object
    position:   x-position of the arrow. If None, mean of xdata is taken
    direction:  'left' or 'right'
    size:       size of the arrow in fontsize points
    color:      if None, line color is taken.
    """
    if color is None:
        color = line.get_color()

    xdata = line.get_xdata()
    ydata = line.get_ydata()

    if position is None:
        position = xdata.mean()
    # find closest index
    start_ind = np.argmin(np.absolute(xdata - position))
    if direction == 'right':
        end_ind = start_ind + 1
    else:
        end_ind = start_ind - 1
    #xend=0.5*(xdata[end_ind]-xdata[start_ind])+xdata[start_ind]
    #yend=0.5*(ydata[end_ind]-ydata[end_ind])+ydata[start_ind]
    if slope is None:
        print('no slope...')
        xend=xdata[end_ind]
        yend=ydata[end_ind]
        xstart=xdata[start_ind]
        ystart=ydata[start_ind]
    else:
        #xend=0.5*(xdata[end_ind]-xdata[start_ind])+xdata[start_ind]
        xend=xdata[end_ind]
        #yend=slope*(xend-xdata[start_ind])+ydata[start_ind] #slope-intercept form that should work
        xstart=0.5*(xdata[start_ind]-xdata[end_ind])+xdata[end_ind]
        ystart=slope*(xstart-xdata[end_ind])+ydata[end_ind] #slope-intercept form that should work
    line.axes.annotate('',
        #xytext=(xdata[start_ind], ydata[start_ind]),
        xytext=(xstart,ystart),
        xy=(xdata[end_ind], ydata[end_ind]),
        #xy=(xend, yend),
        arrowprops=dict(arrowstyle="<|-", color=color),
        size=size
    )




I think I want a more robust errorbar plotting function that will just determine if the input errors actually should be limits and make sure they are pointing the correct direction. So I think I'm going to restructure the input file to use el1/el2_err = -100 to indicate an upper limit and +100 to indicate a lower limit. Then I'll have 

In [10]:
def plot_wd_errorbar(el3el2, el1el2,  el3el2_err, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label='', color='b'):
    if label=='':
        label=name
    else:
        pass
    uplims=False
    lolims=False
    xlolims=False
    xuplims=False
    if np.abs(el3el2_err)> limit_indicator:
        if el3el2_err > 0:
            xlolims=True
        elif el3el2_err < 0:
            xuplims=True
        else:
            print("This shouldn't print el3el2_err")
        el3el2_err=limit_length
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el3el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(el3el2, el1el2, yerr= el1el2_err, xerr= el3el2_err, uplims=uplims, lolims=lolims, xuplims=xuplims, xlolims=xlolims, color=color,marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(el3el2,el1el2, label=fs.fix_display_string(label), marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [11]:

def plot_wd_el1el2el3(row, logg='default', teff='default', t_step=10, naca_min=-4.0, t_max=100, elements=["K","Ca","Na"], t_step_units='Myr',plot_type='line',SSP=True):
    """
    
    
    t_step=10, time in Myr of time-step for declining phase
    
    
    """
    name=row['name']
    print('starting plotting effort for',name)
    string1= elements[0].lower()+'/'+elements[1].lower()
    string2=elements[2].lower()+'/'+elements[1].lower()
    #times= np.arange(0, t_max+t_step, t_step)
    markersize=wd_size
    #target_row=wd_abund_table.loc[name]
    target_row=row
    
    if target_row['show']==0:
        return
    else:
        pass
    target_el1el2=target_row[string1]
    target_el3el2=target_row[string2]
    el1el2_err=target_row[string1+'_err']
    el3el2_err= target_row[string2+'_err']
    label=target_row['name']
    label=fs.fix_display_string(label)
    if logg=='default':
        logg=target_row['logg']
    else:
        pass
    if teff=='default':
        teff= target_row['teff']
    else:
        pass
    #if target_row[string1+'_err']>0.:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=target_row[string1+'_err'], marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #elif target_row[string2+'_err'] < 0.001:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=0.3,yerr=0.3, uplims=True,xuplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #else:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=0.3, uplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    
    plot_wd_errorbar(target_el3el2, target_el1el2,  el3el2_err, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label=label+' Photo.', color=target_row['plot_color'])
    
    
    if SSP:
        
        plot_marker=ssp_marker
        markersize=ci_size
        target_el1el2, target_el3el2, el1el2_err, el3el2_err=acorr.easy_dist_ssp(target_row,elements, plot_all=False,tau_rand=True)
        #label=label+' SSP'
        print('SSP log('+elements[0]+'/' +elements[1]+') =',target_el1el2,'+/-',el1el2_err)
        print('SSP log('+elements[2]+'/' +elements[1]+') =',target_el3el2,'+/-',el3el2_err)
        label=label+' Steady State'
    else:
        print('Not SSP!')
        plot_marker=wd_marker
        el1el2_err=target_row[string2+'_err']
        el3el2_err=target_row[string1+'_err']
        #label=label+' Photo.'
        
    if t_step_units != 'Myr':
        print("t_step_units is not Myr, meaning it's some diffusion timescale multiple")
        print('so t_step= ',t_step,"* tau_",t_step_units)
        tau_time= 10.**itau.extrapolate_tau_x_logg(teff, logg, t_step_units,atm_type=target_row['diff_atm_type'],modeler=acorr.default_modeler, overshoot=acorr.default_overshoot)
        tau_time=tau_time*1e-6 #converted to Myr
        t_step=t_step*tau_time
        t_max=t_max*tau_time
        print('New t_step:',t_step, 'Myr')
        print('New t_max:', t_max, 'Myr')
    else:
        pass
    if target_row['show_dp']==1:
        if plot_type=='line':
            times= np.arange(-1*t_max-t_step, t_max+t_step, t_step)
            #dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row['k/ca'], target_row['na/ca'],times, 'K', "Ca", "Na", logg=logg)
            dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row[string1], target_row[string2],times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            line= plt.plot(dp_el3el2, dp_el1el2, marker='o', label="teff="+str(teff)+'K,logg='+str(logg), color=color_dict[name], alpha=dp_alpha)

            slope=(dp_el1el2-np.roll(dp_el1el2,1))/(dp_el3el2-np.roll(dp_el3el2,1))
            add_arrow(line[0],position=arr_naca[0], slope=slope[0])
            #print('Slope:', slope)
        elif (plot_type=='arrow'):
            print("successfully plot type arrow happening")
            times=t_step
            #arrow_endy,arrow_endx=acorr.el1el2_DP_el3el2_ftimes(teff, target_row[string1], target_row[string2],times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            #print('arrow_endy',arrow_endy,'arrow_endx',arrow_endx)
            #plt.plot(arrow_endx,arrow_endy,marker='o')
            arrow_endy,arrow_endx=acorr.el1el2_DP_el3el2_ftimes(teff, target_el1el2, target_el3el2,times, elements[0], elements[1], elements[2], logg=logg,atm_type=target_row['diff_atm_type'])
            ##ypoints=np.linspace(target_el1el2,arrow_endy,arrow_segs)
            ##xpoints=np.linspace(target_el3el2,arrow_endx,arrow_segs)
            #dx=arrow_endx-target_el3el2
            #dy=arrow_endy-target_el1el2
            ##dx=arrow_endx-xpoints[-2]
            ##dy=arrow_endy-ypoints[-2]
            ##print()
            ##def get_segs(points):
            ##    return np.vstack([points,np.roll(points,1)]).T[1:]
            ##x_segs=get_segs(xpoints)
            ##y_segs=get_segs(ypoints)
            #print('x_segs',x_segs)
            #color_array=np.empty_like(ypoints,dtype=str)
            #color_array[:]=color_dict[name]
            ##alpha_vals=np.linspace(alpha_range[0],alpha_range[1],arrow_segs)
            #print('alpha_vals',alpha_vals)
            #for x,y,alpha in zip(x_segs, y_segs,alpha_vals):
                #plt.plot(x,y,color=color_dict[name],alpha=alpha,linewidth=arrow_line)
                #plt.plot(x,y,color=color_dict[name],alpha=alpha)
            #plt.arrow(xpoints[-2],ypoints[-2],dx,dy,color=color_dict[name],width=arrow_width, alpha=alpha_range[1],length_includes_head=True )
            try:
                #plt.arrow(target_el3el2,target_el1el2,arrow_endx-target_el3el2,arrow_endy-target_el1el2,color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True,linewidth=0 )
                plt.arrow(target_el3el2,target_el1el2,arrow_endx-target_el3el2,arrow_endy-target_el1el2,color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True,linewidth=0 )


            except ValueError:
                print('\nPoint for',label, "can't draw arrow because no data\n")
        else:
            print('\nplot_type not recognized', plot_type,'\n')
    else:
        print('show_dp disabled for object',target_row['show_dp'])
    print('\n\n',target_row['name'])
    #plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    print("about to try")
    #if target_row[elements[0].lower()+'/'+elements[1].lower()+'_err'] > 0.:
    #    plt.errorbar(target_el3el2,target_el1el2, label=fs.fix_display_string(label), marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=el1el2_err, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #elif target_row[elements[2].lower()+'/'+elements[1].lower()+'_err'] < 0.001:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=0.3,yerr=0.3, uplims=True, xuplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    
    #else:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    plot_wd_errorbar(target_el3el2, target_el1el2,  el3el2_err, el1el2_err,name, selected_marker=plot_marker, markersize=markersize, label=label, color=target_row['plot_color'])
    
    print("\n\n***\n\nPlotting concluded for", name,'\n\n***\n')
    return

In [12]:
npoints=100

def plot_objects_el1el2el3(elements=['K','Ca','Na']):
    
    index_val=0
    for row in use_bodies_table:
        object_el1el2=row[elements[0].lower()+'/'+elements[1].lower()]
        object_el3el2=row[elements[2].lower()+'/'+elements[1].lower()]
        #print(row['name'],object_el1el2, object_el3el2)
        #print(type(object_el1el2), object_el1el2, object_el1el2.dtype)
        #print("object_el1el2==False",object_el1el2==False)
        el1el2_clear=False
        el3el2_clear=False
        try:
            object_el1el2.mask
            el1el2_clear=True
        except AttributeError:
            try:
                print(object_el3el2.mask)
                el3el2_clear=True
            except AttributeError:
                #plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                
                if show_all_ssobj_names==True:
                    plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                elif row['show_name']==1:
                    plt.text(object_el3el2, object_el1el2,row['name'],fontsize=figure_text_size)
                else:
                    pass
                
                if "CI" in row['name']:
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color, marker=met_marker, markersize=ci_size, linestyle='None')
                elif index_val==0:
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color,marker=met_marker , label='Solar System Body',linestyle='None',markersize=met_size)
                    index_val+=1


                else:
                    #print('else statement so should be plotting',row['name'],object_el1el2, object_el3el2)
                    plt.errorbar(object_el3el2, object_el1el2, color=met_color, marker=met_marker, linestyle='None',markersize=met_size)
        #index_val+=1
    
    return






In [13]:
def generate_el1el2el3_plot(elements=["K","Ca","Na"],leg_loc='best', show_legend=True,fig_size='default'): 
    spt.initiate_science_plot()

    plot_type='arrow'
    #plot_type='nonsense'
    #plt.figure(figsize=(7.25,7.25),constrained_layout=True)
    if fig_size=='default':
        #plt.figure(figsize=(4.75,4.75),constrained_layout=True)
        plt.figure(figsize=(7.25,7.25),constrained_layout=True)
    else:
        plt.figure(figsize=fig_size,constrained_layout=True)
    #spt.start_ApJ_fig(width_cols=1)




    t_max=10
    count=0
    plot_objects_el1el2el3(elements=elements)


    for j1644_row in wd_abund_table:

        print('\n\n',j1644_row['name'])
        #if count==2:
        #    break
        #t_step=step_dict[j1644_row['name']]
        plot_wd_el1el2el3(j1644_row, logg='default', t_step=t_step, naca_min=-4.0, t_max=t_max, t_step_units='Ca', plot_type=plot_type, elements=elements)
        count+=1
        print("\n\n=====\n",count,"\n====\n\n")
    #plt.errorbar(j1644_row['na/ca'],j1644_row['k/ca'],xerr=j1644_row['na/ca_err'],yerr=0.3, uplims=True,label=j1644_row['name'], marker=wd_marker, color=color_dict[j1644_row['name']], markersize=wd_size)
    print('\n\n***\nWhite dwarf plotting fully concluded\n****\n\n')

    if elements==['Li','Ca',"Na"]:
        plt.xlim(-2.75,1.25)
        plt.ylim(-4.5,0.0)
        #plt.axhline(y=-2.3, linestyle=':',color='k')
        #plt.text(-2.5,-2.25,'GALEXJ2339 photospheric Be-implied spalled Li level from CI')
        #plt.axhline(y=-1.5, linestyle=':',color='k')
        #plt.text(-2.5,-1.45,'GALEXJ2339 photospheric Be-implied spalled Li level from CI')
    elif elements==['K','Ca',"Na"]:
        plt.xlim(-2.75,1.5)
        plt.ylim(-4.25,0.75)
    elif elements==['Li','Na',"K"]:
        plt.xlim(-1.8,0.05)
        plt.ylim(-3.75,-1.0)
    else:
        pass

    #plt.xlabel('log(Na/Ca)')
    #plt.ylabel('log(K/Ca)')
    plt.xlabel('log('+elements[2]+'/'+elements[1]+')')
    plt.ylabel('log('+elements[0]+'/'+elements[1]+')')
    if show_legend:
        plt.legend(loc=leg_loc, fontsize=7)
    else:
        pass
    #plt.xlim(-0.5,0.5)
    #plt.ylim(-1.75,-0.75)
    #plt.xlim(4.5,0)
    #plt.ylim(4.0,1.0)
    

    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        plt.savefig(elements[0]+elements[1]+'_vs_'+elements[2]+elements[1]+'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass

    plt.show()
    return






In [14]:
#generate_el1el2el3_plot(elements=["Na","Li","Ca"])

In [15]:

generate_el1el2el3_plot(elements=["K","Ca","Na"], leg_loc='upper left', show_legend=True)





 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4144.171387479123 7.336853070321065
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4144.171387479123 7.336853070321065
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4144.171387479123 7.336853070321065
tau_K-tau_Ca -0.006871993134912371 +/- 0.19985672224442877
tau_Na-tau_Ca 0.237

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3463.7538901626253 7.6515110871840735
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3463.7538901626253 7.6515110871840735
tau_K-tau_Ca -0.008257410044328856 +/- 0.19756079471478186
tau_Na-tau_Ca 0.24005082737025224 +/- 0.20002934028092106
target_ssp K Ca -1.5904280426836395
target_ssp Na Ca -0.619285414433958
dist ssp K Ca -1.5901625836955273 -1.591742589955671 0.19756079471478188
dist ssp Na Ca -0.6244610323182065 -0.621532938614812 0.30955585520197443
SSP log(K/Ca) = -1.5904280426836395 +/- -100.0
SSP log(Na/Ca) = -0.619285414433958 +/- 0.30955585520197443
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_s

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:76: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [16]:

generate_el1el2el3_plot(elements=["Li","Ca","Na"],leg_loc='upper left')



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3787.501119727671 6.9983618989918845
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3787.501119727671 6.9983618989918845
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3787.501119727671 6.9983618989918845
tau_Li-tau_Ca 0.53917895823474 +/- 0.20690892011279033
tau_Na-tau_Ca 0.2371

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3398.150905048388 7.508193219397153
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3398.150905048388 7.508193219397153
tau_Li-tau_Ca 0.5179286134561135 +/- 0.2005764333218967
tau_Na-tau_Ca 0.23792597524452855 +/- 0.19950173142120883
target_ssp Li Ca -2.5572155543535766
target_ssp Na Ca -0.619285414433958
dist ssp Li Ca -2.5581873484764817 -2.5596331170474147 0.3003845086133076
dist ssp Na Ca -0.6153443079596385 -0.6166674589891018 0.310832255128953
SSP log(Li/Ca) = -2.5572155543535766 +/- 0.3003845086133076
SSP log(Na/Ca) = -0.619285414433958 +/- 0.310832255128953
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
Ne

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:76: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [17]:
#generate_el1el2el3_plot(elements=["Li","Na","K"],fig_size=(5.5,5.5))

In [18]:
#generate_el1el2el3_plot(elements=["Li","Na","Ca"])

In [19]:
#generate_el1el2el3_plot(elements=["Na","Mg","Ca"])

In [20]:
#generate_el1el2el3_plot(elements=["Li","Ca","K"])

In [21]:
#generate_el1el2el3_plot(elements=["Li","K","Na"])

In [22]:
generate_el1el2el3_plot(elements=["Ca","Fe","Mg"],leg_loc='lower right',fig_size=(5.5,5.5))



 WDJ1644-0449
starting plotting effort for WDJ1644-0449
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)
/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:270: UserWarning: Warning: converting a masked element to nan.
  el1el2_dist=np.random.normal(loc=target_el1el2,scale=wd_row[string1+'_err'],size=n_points)
/Users/BenKaiser/Desktop/radial_velocity_calculations/abundance_corrections.py:275: UserWarning: Warning: converting a masked element to nan.
  el3el2_dist=np.random.normal(loc=target_el3el2,scale=wd_row[string2+'_err'],size=n_points)


 He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3451.611188545754 7.418979463427005
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3451.611188545754 7.418979463427005
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3451.611188545754 7.418979463427005
tau_Ca-tau_Fe 0.2051520656427863 +/- 0.20149959674292717
tau_Mg-tau_Fe 0.4491475237461931 +/- 0.20179435676940188
target_ssp Ca Fe --
target_ssp Mg Fe --
dist ssp Ca Fe nan nan nan
dist ssp Mg Fe nan nan nan
SSP log(Ca/Fe) = -- +/- nan
SSP log(Mg/Fe) = -- +/- nan
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koes

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3637.0846226771987 7.4577228050886575
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3637.0846226771987 7.4577228050886575
tau_Ca-tau_Fe 0.2132568433995004 +/- 0.2008610522818044
tau_Mg-tau_Fe 0.4684037688202636 +/- 0.20074590122521604
target_ssp Ca Fe --
target_ssp Mg Fe --
dist ssp Ca Fe nan nan nan
dist ssp Mg Fe nan nan nan
SSP log(Ca/Fe) = -- +/- nan
SSP log(Mg/Fe) = -- +/- nan
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 1479.0021014440508 Myr
New t_max: 2958.0042028881016 Myr
successfully plot type arrow happening
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:76: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [23]:
wd_abund_table.columns

<TableColumns names=('name','modeler','cooling_model','teff','teff_err','logg','logg_err','m_wd','h/he','h/he_err','li/he','li/he_err','na/he','na/he_err','mg/he','mg/he_err','k/he','k/he_err','ca/he','ca/he_err','cr/he','cr/he_err','fe/he','fe/he_err','li/ca','li/ca_err','na/ca','na/ca_err','k/ca','k/ca_err','li/na','li/na_err','k/na','k/na_err','ca/na','ca/na_err','li/k','li/k_err','na/k','na/k_err','ca/fe','ca/fe_err','mg/fe','mg/fe_err','na/mg','na/mg_err','ca/mg','ca/mg_err','ca/cr','ca/cr_err','k/cr','k/cr_err','cr/fe','cr/fe_err','na/li','na/li_err','ca/li','ca/li_err','atm_type','diff_atm_type','log_q','age','age_minus','age_plus','vtan_lsr','vtan_lsr_err_lo','vtan_lsr_err_hi','v','uw2','v_err_lo','v_err_hi','uw2_err_lo','uw2_err_hi','plot_color','show','show_li_evo','show_geo','show_dp','thin_disk','thick_disk','halo')>

In [24]:
np.log10(3.621980492301957*1e6/5)

5.85997610257288

In [25]:
np.log10(17.4506343576774*1e6/5)

6.542841214525449

In [26]:
6.26-5.86

0.39999999999999947

In [27]:
6.56-6.32

0.23999999999999932

In [28]:
0.3999-0.23999

0.15990999999999997

In [29]:
np.log10(9.79e3)-np.log10(2.91e3)

0.5268897028172304

In [30]:
np.log10(4.52e3)-np.log10(2.91e3)

0.19124544582547465

In [31]:
np.log10(2.93e3)-np.log10(2.91e3)

0.0029746313682021963